# JetBot Safety Dataset Collector

Collect 480x480 JPEG images for the desk safety/driving CNN.

Keyboard labels:

- `w`: `img/forward/`
- `a`: `img/turn_left/`
- `d`: `img/turn_right/`
- `s`: `img/stop_reverse/`
- `g`: `img/object/`

Keep the robot stationary while collecting. Move JetBot by hand, then press the label key for the current camera view.

In [1]:
from pathlib import Path
import time

import cv2
import ipywidgets as widgets
from IPython.display import Javascript, display

from jetbot import Camera, bgr8_to_jpeg
from servoserial import ServoSerial

IMAGE_SIZE = 480
IMG_ROOT = Path("img")

KEY_TO_LABEL = {
    "w": "forward",
    "a": "turn_left",
    "d": "turn_right",
    "s": "stop_reverse",
    "g": "object",
}

LABELS = list(KEY_TO_LABEL.values())

for label in LABELS:
    (IMG_ROOT / label).mkdir(parents=True, exist_ok=True)

print("Dataset folders ready:")
for label in LABELS:
    print(" -", IMG_ROOT / label)

Dataset folders ready:
 - img/forward
 - img/turn_left
 - img/turn_right
 - img/stop_reverse
 - img/object


WARNNIG: Jetson.GPIO library has not been verified with this carrier board,


In [3]:
# Fixed camera angle for dataset collection.
servo_device = ServoSerial()
servo_device.Servo_serial_double_control(1, 1000, 2, 1200)
time.sleep(0.5)

camera = Camera.instance(width=IMAGE_SIZE, height=IMAGE_SIZE)

preview = widgets.Image(format="jpeg", width=IMAGE_SIZE, height=IMAGE_SIZE)
status = widgets.HTML()
counts = widgets.HTML()

def numeric_image_ids(label):
    ids = []
    for path in (IMG_ROOT / label).glob("*.jpg"):
        if path.stem.isdigit():
            ids.append(int(path.stem))
    return ids

def next_image_path(label):
    existing = numeric_image_ids(label)
    next_id = max(existing, default=0) + 1
    return IMG_ROOT / label / f"{next_id}.jpg"

def refresh_counts():
    lines = []
    for label in LABELS:
        total = len(list((IMG_ROOT / label).glob("*.jpg")))
        lines.append(f"<b>{label}</b>: {total}")
    counts.value = "<br>".join(lines)

def current_frame_480():
    frame = camera.value
    if frame is None:
        return None
    if frame.shape[0] != IMAGE_SIZE or frame.shape[1] != IMAGE_SIZE:
        frame = cv2.resize(frame, (IMAGE_SIZE, IMAGE_SIZE))
    return frame.copy()

def update_preview(change):
    frame = change["new"]
    if frame is None:
        return
    if frame.shape[0] != IMAGE_SIZE or frame.shape[1] != IMAGE_SIZE:
        frame = cv2.resize(frame, (IMAGE_SIZE, IMAGE_SIZE))
    preview.value = bgr8_to_jpeg(frame)

def capture_label(label):
    if label not in LABELS:
        status.value = f"Invalid label: {label}"
        return

    frame = current_frame_480()
    if frame is None:
        status.value = "No camera frame available"
        return

    path = next_image_path(label)
    ok = cv2.imwrite(str(path), frame)
    if not ok:
        status.value = f"Failed to save: {path}"
        return

    refresh_counts()
    status.value = f"Saved <b>{label}</b>: {path}"

try:
    camera.unobserve(update_preview, names="value")
except Exception:
    pass

camera.observe(update_preview, names="value")

initial = current_frame_480()
if initial is not None:
    preview.value = bgr8_to_jpeg(initial)

buttons = []
for key, label in KEY_TO_LABEL.items():
    button = widgets.Button(description=f"{key}: {label}")
    button.on_click(lambda _, label=label: capture_label(label))
    buttons.append(button)

refresh_counts()
display(preview)
display(widgets.HBox(buttons))
display(status)
display(counts)

print("Camera preview started. Click the notebook output area, then press w/a/s/d/g to capture.")

serial Open!
serial Close!


RuntimeError: Could not initialize camera.  Please see error trace.

In [ ]:
# Keyboard capture hook for classic Jupyter Notebook.
# If keys do not work, use the buttons above as a fallback.
display(Javascript(r'''
(function() {
  if (window.jetbotDatasetKeyHandler) {
    document.removeEventListener('keydown', window.jetbotDatasetKeyHandler);
  }

  const keyToLabel = {
    w: 'forward',
    a: 'turn_left',
    d: 'turn_right',
    s: 'stop_reverse',
    g: 'object'
  };

  window.jetbotDatasetKeyHandler = function(event) {
    const target = event.target || {};
    const tagName = (target.tagName || '').toLowerCase();
    if (tagName === 'input' || tagName === 'textarea' || target.isContentEditable) {
      return;
    }

    const key = (event.key || '').toLowerCase();
    const label = keyToLabel[key];
    if (!label || event.repeat) {
      return;
    }

    event.preventDefault();
    const code = 'capture_label(' + JSON.stringify(label) + ')';
    if (window.Jupyter && Jupyter.notebook && Jupyter.notebook.kernel) {
      Jupyter.notebook.kernel.execute(code);
    } else {
      console.warn('Jupyter kernel object not found. Use the on-screen buttons.');
    }
  };

  document.addEventListener('keydown', window.jetbotDatasetKeyHandler);
  console.log('JetBot dataset keyboard handler attached: w/a/s/d/g');
})();
'''))

status.value = "Keyboard handler attached: w/a/s/d/g"

In [ ]:
# Cleanup cell. Run this before restarting or opening another camera notebook.
try:
    camera.unobserve(update_preview, names="value")
except Exception as exc:
    print("camera unobserve skipped:", exc)

try:
    camera.stop()
except Exception as exc:
    print("camera stop skipped:", exc)

display(Javascript(r'''
(function() {
  if (window.jetbotDatasetKeyHandler) {
    document.removeEventListener('keydown', window.jetbotDatasetKeyHandler);
    window.jetbotDatasetKeyHandler = null;
    console.log('JetBot dataset keyboard handler removed');
  }
})();
'''))

print("Dataset collector stopped")